In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


import copy

import gymnasium as gym
import matplotlib.pyplot as plt

from torch.utils.tensorboard import SummaryWriter

# import sys, os
# sys.path.append(os.getcwd()+"/../src")
from mcts_torch import MCTSNode, MCTSTree

# for vscode debugging view
normal_repr = torch.Tensor.__repr__
torch.Tensor.__repr__ = lambda self: f"{self.shape}_{normal_repr(self)}"

In [2]:
def transform_value_reward_large_to_small(x:torch.Tensor, eps = 0.001): # large to small
    out = torch.sign(x) * (torch.sqrt(torch.abs(x) +1) - 1 + eps*x)
    assert out.shape == x.shape
    return out

def transform_value_reward_small_to_large(y:torch.Tensor, eps=0.001): # small to large
    root = torch.sqrt(y/eps + 1 + torch.sign(y)/eps + (1/(2*eps))**2)
    out = torch.sign(y)*((root-1/(2*eps))**2 - 1)
    assert y.shape == out.shape
    return out

In [3]:
class DummyRepresentationModel(nn.Module): # s0 = h(past_observations)
    def __init__(self, dim_obs_in, n_hidden, n_hidden_state): # TODO encode several past observations, TODO encode actions
        super().__init__()
        
        self.dim_obs_in = dim_obs_in

        self.hidden_state_net = nn.Sequential(
            nn.Linear(dim_obs_in, n_hidden),
            nn.Tanh(),
            nn.Linear(n_hidden, n_hidden_state),
        )

    def forward(self, obs): # TODO encode action in input to dynamics function
        if not isinstance(obs, torch.Tensor): obs = torch.tensor(obs)
        hidden_state = self.hidden_state_net(obs)
        return hidden_state

class DummyPredictionModel(nn.Module): # p_k, v_k = f(hidden)
    def __init__(self, n_hidden_state, n_actions, range_from=-25, range_to=25):
        super().__init__()

        # self.prediction_net_body = nn.Sequential(
        #     nn.
        # )
        self.n_actions = n_actions
        self.range_from = range_from
        self.range_to = range_to

        self.prediction_net_policy_head = nn.Sequential(
            nn.Tanh(),
            nn.Linear(in_features=n_hidden_state, out_features=n_hidden_state),
            nn.Tanh(),
            nn.Linear(in_features=n_hidden_state, out_features=self.n_actions)
        )

        # TODO this is initialized as roughly the average of the output range transformed to larger scale, e.g [0, 25] range initialized value will be ~ 175
        # Better if it was initialized to match an initial reward (avoid long burn-in time)
        self.prediction_net_value_head = nn.Sequential(
            nn.Tanh(),
            nn.Linear(in_features=n_hidden_state, out_features=n_hidden_state),
            nn.Tanh(),
            nn.Linear(in_features=n_hidden_state, out_features=self.range_to - self.range_from + 1)
        )

    def forward(self, hidden_state):
        if not isinstance(hidden_state, torch.Tensor): hidden_state = torch.tensor(hidden_state)
        # if hidden_state.ndim == 1: hidden_state = hidden_state.unsqueeze(0) # add batch dim
        #_prediction_body   = self.prediction_net_body(hidden_state)
        _prediction_body = hidden_state
        policy_logits_pred = self.prediction_net_policy_head(_prediction_body)
        value_logits_pred  = self.prediction_net_value_head(_prediction_body)
        # return policy_logits_pred, 50.0*value_logits_pred 
        if self.training:
            return policy_logits_pred, value_logits_pred
        else:
            return policy_logits_pred, transform_value_reward_small_to_large(F.softmax(value_logits_pred, dim=-1) @ torch.arange(self.range_from, self.range_to + 1).to(dtype=value_logits_pred.dtype))

class DummyDynamicsModel(nn.Module): # r_k, s_k = g(skm1, a_k) # reward, new hidden state
    def __init__(self, n_hidden_state, n_actions, range_from=-10, range_to=10):
        super().__init__()
        self.n_actions = n_actions
        self.range_from = range_from
        self.range_to = range_to
        # rk, sk = g(skm1, ak) # reward, new hidden state
        # TODO encode action
        self.dynamics_net_body = nn.Sequential(
            nn.Tanh(),
            nn.Linear(in_features=n_hidden_state+1, out_features=n_hidden_state) # down-sampling
        )

        self.dynamics_net_reward_head = nn.Sequential(
            nn.Tanh(),
            # nn.Linear(in_features=n_hidden_state, out_features=1)
            nn.Linear(in_features=n_hidden_state, out_features=self.range_to - self.range_from + 1)
        )

        self.dynamics_net_next_state_head = nn.Sequential(
            nn.Tanh(),
            nn.Linear(in_features=n_hidden_state, out_features=n_hidden_state),
        )

    def forward(self, hidden_state, action): 
        # discrete action -> TODO transform to [0, 1] by dividing by n_actions (or 1+n_actions)
        # output batchdim is given by hidden_state batchdim
        if not isinstance(hidden_state, torch.Tensor): hidden_state = torch.tensor(hidden_state)
        if not isinstance(action, torch.Tensor): action = torch.tensor(action)
        while action.ndim < hidden_state.ndim: action = action.unsqueeze(action.ndim) # action will have same dim as hidden_state
        # either (N, h) and (N, 1) or (h) and (1)
        assert action.ndim == hidden_state.ndim
        if action.ndim == 2: assert action.shape[0] == hidden_state.shape[0]
        hidden_state = torch.cat([hidden_state, action], dim=hidden_state.ndim-1)
        _dynamics_body   = self.dynamics_net_body(hidden_state)
        reward_logits_pred = self.dynamics_net_reward_head(_dynamics_body)
        next_state_pred  = self.dynamics_net_next_state_head(_dynamics_body)
        if self.training:
            return reward_logits_pred, next_state_pred
        else:
            reward = transform_value_reward_small_to_large(F.softmax(reward_logits_pred, dim=-1) @ torch.arange(self.range_from, self.range_to + 1).to(dtype=reward_logits_pred.dtype))
            return reward, next_state_pred
        # return reward_logits_pred, next_state_pred # TODO maybe reward head is not complex enough to predict 1 most of the time, but 0 sometimes

In [4]:
# Reward loss (should be cross entropy)
def l_r(input, target):
    # return torch.mean(torch.square(input - target), dim=1)
    assert input.shape == target.shape
    out = torch.square(input - target)
    assert out.shape == input.shape
    return out

# Value loss (should be cross entropy)
def l_v(input, target):
    assert input.shape == target.shape
    out = torch.mean(torch.square(input - target), dim=1)
    assert out.shape == input.shape
    return out

# Policy loss (cross entropy)
def l_p(input_logits, target_probs):
    """Policy loss function"""
    assert input_logits.shape == target_probs.shape
    had_batch_dim = input_logits.ndim == 2
    out = nn.CrossEntropyLoss(reduction='none')(input_logits, target_probs)
    out = out.unsqueeze(out.ndim)
    if had_batch_dim:
        assert out.shape[0] == input_logits.shape[0]
        assert out.shape[1] == 1
    else:
        out.ndim == input_logits.ndim
        out.shape[0] == 1
    return out

def l_p_custom(input_logits, target_probs):
    assert input_logits.shape == target_probs.shape
    had_batch_dim = input_logits.ndim == 2
    out = -(target_probs * input_logits).sum(dim=-1)
    out = out.unsqueeze(out.ndim)
    if had_batch_dim:
        assert out.shape[0] == input_logits.shape[0]
        assert out.shape[1] == 1
    else:
        out.ndim == input_logits.ndim
        out.shape[0] == 1
    return out

In [5]:
# TODO input x of shape [N, 1] or [1, 1] or [1] or []
def num_to_support(x:torch.Tensor, range_from = -10, range_to = 10):
    assert x.ndim == 2, 'currently only for tensors with batch dim'
    assert x.shape[1] == 1, 'currently only for tensors of shape [N, 1]'
    N = x.shape[0]
    y = x.squeeze().clamp(range_from, range_to)
    l = torch.floor(y).to(dtype=torch.int)
    u = l+1
    wl = u - y
    wu = y - l
    # x = wl * l + wu * u
    out_weight_vec = torch.zeros(size=(N, range_to - range_from + 1), dtype=x.dtype)
    out_weight_vec[torch.arange(N), l-range_from] = wl
    upperidx = torch.minimum(u-range_from, torch.tensor(out_weight_vec.shape[1])-1)
    upperweight = torch.where(u-range_from == out_weight_vec.shape[1], wl, wu)
    out_weight_vec[torch.arange(N), upperidx] = upperweight
    assert out_weight_vec.shape[0] == x.shape[0]
    return out_weight_vec

In [6]:
# Setup

env_name = "CartPole-v1"
env = gym.make(env_name)


gamma = 0.999
alpha = 0.001 # TODO this is not given in the paper -> look up in pseudocode
buffer_size = 200
mini_batch_size = 50
n_train_iterations = 1 # how many mini batches are sampled from the buffer
n_MCTS_sims_data_generation = 5
use_MCTS_in_evaluation = True
n_MCTS_sims_evaluation = 5
c = 0.001 # L2 penalty (used in Adam weight_decay)
n_hidden_state = 32

reward_from = -10 # reward and value discretization range, but after resizing with the invertible transformation
reward_to =  10
value_from = -25
value_to = 25

n_actions = int(env.action_space.n)

h = DummyRepresentationModel(dim_obs_in=env.observation_space.shape[0], n_hidden=32, n_hidden_state=n_hidden_state)
f = DummyPredictionModel(n_hidden_state, n_actions=n_actions, range_from=value_from, range_to=value_to)
g = DummyDynamicsModel(n_hidden_state=n_hidden_state, n_actions=n_actions, range_from=reward_from, range_to=reward_to)

h.train()
f.train()
g.train()

optimizer = torch.optim.Adam(params=list(h.parameters()) + list(f.parameters()) + list(g.parameters()), lr=alpha, weight_decay=c)

In [7]:
writer = SummaryWriter(log_dir=f"../runs2/MuZero_{env_name}")
hparam_dict={
    'gamma' : gamma,
    'alpha' : alpha, # TODO this is not given in the paper -> look up in pseudocode
    'buffer_size' : buffer_size,
    'mini_batch_size' : mini_batch_size,
    'n_train_iterations' : n_train_iterations, # how many mini batches are sampled from the buffer
    'n_MCTS_sims_data_generation' : n_MCTS_sims_data_generation,
    'use_MCTS_in_evaluation' : use_MCTS_in_evaluation,
    'n_MCTS_sims_evaluation' : n_MCTS_sims_evaluation,
    'c' : c, # L2 penalty (used in Adam weight_decay)
    'n_hidden_state' : n_hidden_state
}
metric_dict={}
writer.add_hparams(hparam_dict=hparam_dict,metric_dict=metric_dict)

In [8]:
buffer_S          = torch.empty(size=(buffer_size, *env.observation_space.shape), dtype=torch.float32)
buffer_A          = torch.empty(size=(buffer_size, 1), dtype=torch.int32)
buffer_R          = torch.empty(size=(buffer_size, 1), dtype=torch.float32)
buffer_S_prime    = torch.empty(size=(buffer_size, *env.observation_space.shape), dtype=torch.float32)
buffer_terminated = torch.empty(size=(buffer_size, 1), dtype=torch.bool)
buffer_truncated  = torch.empty(size=(buffer_size, 1), dtype=torch.bool)
buffer_nu         = torch.empty(size=(buffer_size, 1), dtype=torch.float32)
buffer_pi         = torch.empty(size=(buffer_size, n_actions), dtype=torch.float32)

In [9]:
# Generate experience with saved data_generation_model
# Refresh entire buffer

i_epoch = -1

# while True:
#     i_epoch +=1

h_checkpoint = copy.deepcopy(h)
f_checkpoint = copy.deepcopy(f)
g_checkpoint = copy.deepcopy(g)
for model in [h_checkpoint, f_checkpoint, g_checkpoint]:
    model.eval()
    for param in model.parameters():
        param.requires_grad = False

S, info = env.reset()
done = False

for istep in range(buffer_size):

    with torch.no_grad():
        hidden_state                          = h_checkpoint(torch.tensor(S, dtype=torch.float32))
        policy_logits_pred, value_logits_pred = f_checkpoint(hidden_state)

        tree = MCTSTree(
            MCTSNode(
                hidden_state=hidden_state,
                n_actions=n_actions,
                p=F.softmax(policy_logits_pred, dim=-1),
                v = value_logits_pred,
                identifier="root",
                ),
            n_simulations=n_MCTS_sims_data_generation,
            dynamics_function=g_checkpoint,
            prediction_function=f_checkpoint,
            n_actions=n_actions
        )
        nu, pi = tree.search()

        A = tree.sample()

    S_prime, R, terminated, truncated, info = env.step(A.item())

    # Store in buffer
    # for now, save states, actions, rewards, next states, truncated/terminated
    # Calculate n-step returns later
    # Re-calculate or update save method for any values of the play_network
    buffer_S[istep, ...]          = torch.tensor(S, dtype=buffer_S.dtype)
    buffer_A[istep, ...]          = A.to(dtype=buffer_A.dtype)
    buffer_R[istep, ...]          = torch.tensor(R, dtype=buffer_R.dtype)
    buffer_S_prime[istep, ...]    = torch.tensor(S_prime, dtype=buffer_S_prime.dtype)
    buffer_terminated[istep, ...] = torch.tensor(terminated, dtype=buffer_terminated.dtype)
    buffer_truncated[istep, ...]  = torch.tensor(truncated, dtype=buffer_truncated.dtype)
    
    buffer_nu[istep, ...] = nu
    buffer_pi[istep, ...] = pi

    done = terminated or truncated

    if done:
        S, info = env.reset()
        done = False
    else:
        S = S_prime
    

    # writer.add_scalar('MCTS/v',     value_logits_pred[0].detach().mean().item(),     i_epoch*buffer_size + istep)
    # writer.add_scalar('MCTS/nu',     nu[0].detach().mean().item(),     i_epoch*buffer_size + istep)
    writer.add_scalars('MCTS/values', {'v': value_logits_pred.detach().mean().item(), 'nu': nu.detach().mean().item()}, i_epoch*buffer_size + istep)
    # TODO fix why do the dictionaries not show up in tensorboard
    writer.add_scalars('MCTS/p',     {'0': F.softmax(policy_logits_pred, dim=-1)[0].detach().item(), '1': F.softmax(policy_logits_pred, dim=-1)[1].detach().item()},     i_epoch*buffer_size + istep)
    writer.add_scalars('MCTS/pi',    {'0': pi[0].detach().item(), '1': pi[1].detach().item()},     i_epoch*buffer_size + istep)

In [19]:
# sample a mini batch, calc and train
# TODO prioritized sampling
# Reevaluated model: hidden_state, policy_logits_pred, value_logits_pred, reward_logits_pred, next_state_pred
# TODO this is pseudo code: this should be a sum over k, K successive steps, evaluations of the current model along 5 steps of generated data
# TODO rename R to u, and reward predictions to r
# TODO compute z for every state in the buffer, then compute p=abs(nu - z) for every state, then do prioritized sampling
 
K = 2
n = 3

h.train()
f.train()
g.train()

for i_iteration in range(n_train_iterations):
    ixs = np.random.default_rng().choice(buffer_size-K-2, mini_batch_size, replace=False) # -K+1 so that we don't access elements out of bounds # TODO for n=10 we need max(K, 10) successive entries from the buffer

    mb_R1 = buffer_R[ixs, ...]
    mb_R2 = buffer_R[ixs+1, ...]
    mb_R3 = buffer_R[ixs+2, ...]
    mb_R4 = buffer_R[ixs+3, ...]
    
    mb_terminated1 = buffer_terminated[ixs, ...]
    mb_truncated1 = buffer_truncated[ixs, ...]
    mb_terminated2 = buffer_terminated[ixs+1, ...]
    mb_truncated2 = buffer_truncated[ixs+1, ...]
    mb_terminated3 = buffer_terminated[ixs+2, ...]
    mb_truncated3 = buffer_truncated[ixs+2, ...]
    mb_terminated4 = buffer_terminated[ixs+3, ...]
    mb_truncated4 = buffer_truncated[ixs+3, ...]
    
    mb_S1 = buffer_S[ixs, ...]
    # mb_S2 = buffer_S[ixs+1, ...] # delete if no error
    # mb_S_prime1 = buffer_S_prime[ixs, ...]
    # mb_S_prime2 = buffer_S_prime[ixs+1, ...]
    
    mb_A1 = buffer_A[ixs, ...]
    mb_A2 = buffer_A[ixs+1, ...]

    mb_As = torch.cat([buffer_A[ixs+i, :] for i in range(K)], dim=1) # shape (mini_batch_size, K)
    assert mb_As.shape == (mini_batch_size, K)
    assert (mb_A1.squeeze() == mb_As[:, 0].squeeze()).all()
    assert (mb_A2.squeeze() == mb_As[:, 1].squeeze()).all() 

    
    mb_nu2 = buffer_nu[ixs+1, ...] # for the next state
    mb_nu3 = buffer_nu[ixs+1+1, ...] # the next-next state
    mb_nu4 = buffer_nu[ixs+1+2, ...] # the next-next state
    mb_nu5 = buffer_nu[ixs+1+3, ...] # the next-next state
    
    mb_pi1 = buffer_pi[ixs, ...]
    mb_pi2 = buffer_pi[ixs+1, ...]

    mb_pis = torch.stack([buffer_pi[ixs+i, :] for i in range(K)], dim=2) # shape (mini_batch_size, n_actions, K)
    assert mb_pis.shape == (mini_batch_size, n_actions, K)
    assert (mb_pi1.squeeze() == mb_pis[:, :, 0].squeeze()).all()
    assert (mb_pi2.squeeze() == mb_pis[:, :, 1].squeeze()).all()
    

    indices = torch.hstack([torch.tensor(ixs).reshape((mini_batch_size, 1))+i for i in range(n+K-1)])
    
    mb_Rs = torch.take(buffer_R, indices)
    assert mb_Rs.shape == (mini_batch_size, K-1+n)
    assert (mb_Rs[:, 0] == mb_R1.squeeze()).all()
    assert (mb_Rs[:, 1] == mb_R2.squeeze()).all()
    assert (mb_Rs[:, 2] == mb_R3.squeeze()).all()
    assert (mb_Rs[:, 3] == mb_R4.squeeze()).all()
    
    mb_terminateds = torch.take(buffer_terminated, indices)
    assert mb_terminateds.shape == (mini_batch_size, K-1+n)
    assert (mb_terminateds[:, 0] == mb_terminated1.squeeze()).all()
    assert (mb_terminateds[:, 1] == mb_terminated2.squeeze()).all()
    assert (mb_terminateds[:, 2] == mb_terminated3.squeeze()).all()
    assert (mb_terminateds[:, 3] == mb_terminated4.squeeze()).all()
    
    mb_truncateds = torch.take(buffer_truncated, indices)
    assert mb_truncateds.shape == (mini_batch_size, K-1+n)
    assert (mb_truncateds[:, 0] == mb_truncated1.squeeze()).all()
    assert (mb_truncateds[:, 1] == mb_truncated2.squeeze()).all()
    assert (mb_truncateds[:, 2] == mb_truncated3.squeeze()).all()
    assert (mb_truncateds[:, 3] == mb_truncated4.squeeze()).all()

    
    indices_nus = indices + 1 # saved at the next state
    # indices_nus = indices_nus[:, -K:] # only need K nus
    mb_nus = torch.take(buffer_nu, indices_nus)
    assert mb_nus.shape == (mini_batch_size, K+n-1)
    assert (mb_nus[:, 2] == mb_nu4.squeeze()).all()
    assert (mb_nus[:, 3] == mb_nu5.squeeze()).all()


    # Treatment of terminal states as absorbing states
    # reward 0, value 0, uniform policy, action random (of all subsequent states TODO)
    # TODO distinguish terminated and truncated
    mb_terminal1 = torch.logical_or(mb_terminated1, mb_truncated1)
    mb_terminal2 = torch.logical_or(mb_terminated2, mb_truncated2)
    mb_terminal3 = torch.logical_or(mb_terminated3, mb_truncated3)
    mb_terminal4 = torch.logical_or(mb_terminated4, mb_truncated4)

    mb_terminals = torch.logical_or(mb_terminateds, mb_truncateds)
    assert (mb_terminals[:, 0] == mb_terminal1.squeeze()).all()
    assert (mb_terminals[:, 1] == mb_terminal2.squeeze()).all()
    assert (mb_terminals[:, 2] == mb_terminal3.squeeze()).all()
    assert (mb_terminals[:, 3] == mb_terminal4.squeeze()).all()

    
    mb_R2 = torch.where(mb_terminal1, torch.zeros_like(mb_R2), mb_R2)
    mb_R3 = torch.where(mb_terminal1, torch.zeros_like(mb_R3), mb_R3)
    mb_R4 = torch.where(mb_terminal1, torch.zeros_like(mb_R4), mb_R4)
        
    mb_R3 = torch.where(mb_terminal2, torch.zeros_like(mb_R3), mb_R3)
    mb_R4 = torch.where(mb_terminal2, torch.zeros_like(mb_R4), mb_R4)
    
    mb_R4 = torch.where(mb_terminal3, torch.zeros_like(mb_R4), mb_R4)
    
    for j in range(mb_terminals.shape[1]-1):
        for k in range(1, mb_Rs.shape[1]-j):
            mb_Rs[:, j+k] = torch.where(mb_terminals[:, j], torch.zeros_like(mb_Rs[:, j+k]), mb_Rs[:, j+k])

    assert (mb_R1.squeeze() == mb_Rs[:, 0]).all()
    assert (mb_R2.squeeze() == mb_Rs[:, 1]).all()
    assert (mb_R3.squeeze() == mb_Rs[:, 2]).all()
    assert (mb_R4.squeeze() == mb_Rs[:, 3]).all()

    
    mb_nu2 = torch.where(mb_terminal1, torch.zeros_like(mb_nu2), mb_nu2)
    mb_nu3 = torch.where(mb_terminal1, torch.zeros_like(mb_nu3), mb_nu3)
    mb_nu4 = torch.where(mb_terminal1, torch.zeros_like(mb_nu4), mb_nu4)
    mb_nu5 = torch.where(mb_terminal1, torch.zeros_like(mb_nu5), mb_nu5)

    mb_nu3 = torch.where(mb_terminal2, torch.zeros_like(mb_nu3), mb_nu3)
    mb_nu4 = torch.where(mb_terminal2, torch.zeros_like(mb_nu4), mb_nu4)
    mb_nu5 = torch.where(mb_terminal2, torch.zeros_like(mb_nu5), mb_nu5)

    mb_nu4 = torch.where(mb_terminal3, torch.zeros_like(mb_nu4), mb_nu4)
    mb_nu5 = torch.where(mb_terminal3, torch.zeros_like(mb_nu5), mb_nu5)

    mb_nu5 = torch.where(mb_terminal4, torch.zeros_like(mb_nu5), mb_nu5)


    for i_terminal in range(mb_terminals.shape[1]):
        for i_nu in range(i_terminal, mb_nus.shape[1]):
            mb_nus[:, i_nu] = torch.where(mb_terminals[:, i_terminal], torch.zeros_like(mb_nus[:, i_nu]), mb_nus[:, i_nu])

    assert (mb_nu4.squeeze() == mb_nus[:, 2]).all()
    assert (mb_nu5.squeeze() == mb_nus[:, 3]).all()
    
    mb_nus = mb_nus[:, -K:] # TODO get the indexing right so we only need K nus from the start


    # TODO I don't know if it is correct to modify the MCTS target policy at states following a terminal state, do ablation?
    mb_pi2 = torch.where(mb_terminal1, torch.ones(size=(mini_batch_size, n_actions))/n_actions, mb_pi2)
    for i_terminal in range(K):
        for i_pi in range(i_terminal+1, K):
            # mb_pis[:, :, i_pi] = torch.where(mb_terminals[:, i_terminal], torch.ones(size=(mini_batch_size, n_actions))/n_actions, mb_pis[:, :, i_pi])
            # TODO why are the shapes so weird
            where_condition = mb_terminals[:, i_terminal].reshape((mini_batch_size, 1)).expand((mini_batch_size, n_actions))
            mb_pis[:, :, i_pi] = torch.where(where_condition, torch.ones(size=(mini_batch_size, n_actions))/n_actions, mb_pis[:, :, i_pi])
    assert (mb_pis[:, :, 1] == mb_pi2).all()
    assert (mb_pis[:, :, 0] == mb_pi1).all() # should stay unchanged

    generator = torch.Generator().manual_seed(123)
    mb_A2 = torch.where(mb_terminal1, torch.multinomial(mb_pi2, num_samples=1, generator=generator), mb_A2)
    for i_terminal in range(K):
        for i_A in range(i_terminal+1, K):
            generator = torch.Generator().manual_seed(123)
            mb_As[:, i_A] = torch.where(mb_terminals[:, i_terminal], torch.multinomial(mb_pis[:, :, i_A], num_samples=1, generator=generator).squeeze(), mb_As[:, i_A])
    assert mb_As.shape == (mini_batch_size, K)
    assert (mb_As[:, 1] == mb_A2.squeeze()).all()


    mb_hidden_state1 = h(mb_S1)

    mb_p_pred1, mb_v_pred1_logits = f(mb_hidden_state1)
    mb_R_pred1_logits, mb_hidden_state2 = g(mb_hidden_state1, mb_A1)

    mb_p_pred2, mb_v_pred2_logits = f(mb_hidden_state2)
    mb_R_pred2_logits, mb_hidden_state3 = g(mb_hidden_state2, mb_A2)


    mb_hidden_states = torch.zeros((mini_batch_size, n_hidden_state, K+1))
    mb_p_preds = torch.zeros((mini_batch_size, n_actions, K))
    mb_v_preds_logits = torch.zeros((mini_batch_size, value_to - value_from + 1, K))
    mb_R_preds_logits = torch.zeros((mini_batch_size, reward_to - reward_from + 1, K))

    mb_hidden_states[:, :, 0] = h(mb_S1)
    for k in range(K):
        mb_p_preds[:, :, k], mb_v_preds_logits[:, :, k] = f(mb_hidden_states[:, :, k])
        mb_R_preds_logits[:, :, k], mb_hidden_states[:, :, k+1] = g(mb_hidden_states[:, :, k], mb_As[:, k])
    assert (mb_hidden_state1 == mb_hidden_states[:, :, 0]).all()
    assert (mb_hidden_state2 == mb_hidden_states[:, :, 1]).all()
    assert (mb_hidden_state3 == mb_hidden_states[:, :, 2]).all() # there may be some randomness due to setting A to a random value after termination
    assert (mb_p_pred1 == mb_p_preds[:, :, 0]).all()
    assert (mb_p_pred2 == mb_p_preds[:, :, 1]).all()
    assert (mb_v_pred1_logits == mb_v_preds_logits[:, :, 0]).all()
    assert (mb_v_pred2_logits == mb_v_preds_logits[:, :, 1]).all()
    assert (mb_R_pred1_logits == mb_R_preds_logits[:, :, 0]).all()
    assert (mb_R_pred2_logits == mb_R_preds_logits[:, :, 1]).all()


    # For diagnostics, can leave 2 values
    mb_R_pred1 = transform_value_reward_small_to_large(F.softmax(mb_R_pred1_logits, dim=-1) @ torch.arange(reward_from, reward_to+1).to(dtype=mb_R_pred1_logits.dtype))
    mb_R_pred2 = transform_value_reward_small_to_large(F.softmax(mb_R_pred2_logits, dim=-1) @ torch.arange(reward_from, reward_to+1).to(dtype=mb_R_pred2_logits.dtype))

    mb_v_pred1 = transform_value_reward_small_to_large(F.softmax(mb_v_pred1_logits, dim=-1) @ torch.arange(value_from, value_to+1).to(dtype=mb_v_pred1_logits.dtype))
    mb_v_pred2 = transform_value_reward_small_to_large(F.softmax(mb_v_pred2_logits, dim=-1) @ torch.arange(value_from, value_to+1).to(dtype=mb_v_pred2_logits.dtype))

    mb_z1 = mb_R1 + gamma * mb_R2 + gamma**2 * mb_R3 + gamma**3 * mb_nu4 # n+1 terms
    mb_z2 = mb_R2 + gamma * mb_R3 + gamma**2 * mb_R4 + gamma**3 * mb_nu5 
    
    
    mb_zs = torch.zeros((mini_batch_size, K))
    for i_zs in range(K):
        for i_Rs in range(n):
            mb_zs[:, i_zs] += gamma**i_Rs * mb_Rs[:, i_Rs+i_zs] # mb_R1 + gamma * mb_R2 + gamma**2 * mb_R3 + gamma**3 * mb_nu4
        mb_zs[:, i_zs] += gamma**n * mb_nus[:, i_zs]
    
    assert mb_zs.shape == (mini_batch_size, K)
    assert (mb_z1.squeeze() == mb_zs[:, 0]).all()
    assert (mb_z2.squeeze() == mb_zs[:, 1]).all()

    # Losses
    # K terms
    mb_loss_R1 = l_p(mb_R_pred1_logits, num_to_support(transform_value_reward_large_to_small(mb_R1), range_from=reward_from, range_to=reward_to))
    mb_loss_R2 = l_p(mb_R_pred2_logits, num_to_support(transform_value_reward_large_to_small(mb_R2), range_from=reward_from, range_to=reward_to))
    
    mb_loss_v1 = l_p(mb_v_pred1_logits, num_to_support(transform_value_reward_large_to_small(mb_z1), range_from=value_from, range_to=value_to))
    mb_loss_v2 = l_p(mb_v_pred2_logits, num_to_support(transform_value_reward_large_to_small(mb_z2), range_from=value_from, range_to=value_to))

    mb_loss_p1 = l_p(mb_p_pred1, mb_pi1)
    mb_loss_p2 = l_p(mb_p_pred2, mb_pi2)

    assert mb_loss_R1.shape == mb_loss_v1.shape == mb_loss_p1.shape == mb_loss_R2.shape == mb_loss_v2.shape == mb_loss_p2.shape

    mb_loss_R = torch.zeros((mini_batch_size, 1))
    mb_loss_v = torch.zeros((mini_batch_size, 1))
    mb_loss_p = torch.zeros((mini_batch_size, 1))
    for k in range(K):
        mb_loss_R += l_p(mb_R_preds_logits[:, :, k], num_to_support(transform_value_reward_large_to_small(mb_Rs[:, k, torch.newaxis]), range_from=reward_from, range_to=reward_to))
        mb_loss_v += l_p(mb_v_preds_logits[:, :, k], num_to_support(transform_value_reward_large_to_small(mb_zs[:, k, torch.newaxis]), range_from=value_from, range_to=value_to))
        mb_loss_p += l_p(mb_p_preds[:, :, k], mb_pis[:, :, k])
    
    assert (mb_loss_R == mb_loss_R1 + mb_loss_R2).all()
    assert (mb_loss_v == mb_loss_v1 + mb_loss_v2).all()
    assert (mb_loss_p == mb_loss_p1 + mb_loss_p2).all()

    mb_loss1 = mb_loss_R1 + mb_loss_v1 + mb_loss_p1
    mb_loss2 = mb_loss_R2 + mb_loss_v2 + mb_loss_p2
    
    loss = (1.0/K)*(mb_loss1.mean() + mb_loss2.mean())

    loss2 = (mb_loss_R + mb_loss_v + mb_loss_p).sum()/(K*mini_batch_size) # TODO replace scaling by K with gradient scaling

    assert torch.allclose(loss, loss2)
    # Train one step
    for model in [h, f, g]:
        for param in model.parameters():
            param.grad = None
    loss.backward()
    optimizer.step()

    writer.add_scalar('Loss/loss',   loss.detach().item(),                           i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Loss/mb_R1',  mb_loss_R1.detach().mean().item(),  i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Loss/mb_R2',  mb_loss_R2.detach().mean().item(),  i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Loss/mb_z1',  mb_loss_v1.detach().mean().item(),  i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Loss/mb_z2',  mb_loss_v1.detach().mean().item(),  i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Loss/mb_pi1', mb_loss_p1.detach().mean().item(), i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Loss/mb_pi2', mb_loss_p1.detach().mean().item(), i_epoch*n_train_iterations + i_iteration)

    writer.add_scalar('Values1/mb_R_pred1', mb_R_pred1[0].detach().item(), i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values1/mb_R1',      mb_R1[0].detach().item(),      i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values1/mb_v_pred1', mb_v_pred1[0].detach().item(), i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values1/mb_z1',      mb_z1[0].detach().item(),      i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values1/mb_p_pred1', mb_p_pred1[0, 0].detach().item(), i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values1/mb_pi1',     mb_pi1[0, 0].detach().item(),     i_epoch*n_train_iterations + i_iteration)
    
    writer.add_scalar('Values2/mb_R_pred2', mb_R_pred2[0].detach().item(), i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values2/mb_R2',      mb_R2[0].detach().item(),      i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values2/mb_v_pred2', mb_v_pred2[0].detach().item(), i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values2/mb_z2',      mb_z2[0].detach().item(),      i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values2/mb_p_pred2', mb_p_pred2[0, 0].detach().item(), i_epoch*n_train_iterations + i_iteration)
    writer.add_scalar('Values2/mb_pi2',     mb_pi2[0, 0].detach().item(),     i_epoch*n_train_iterations + i_iteration)

In [18]:
torch.allclose(loss, loss2)

True

In [ ]:
generator=torch.Generator().manual_seed(123)
a = torch.multinomial(mb_pi2, num_samples=1, generator=generator)
generator=torch.Generator().manual_seed(123)
b = torch.multinomial(mb_pis[:, :, i_A], num_samples=1, generator=generator)
(a == b).all()

torch.Size([])_tensor(True)

torch.Size([50, 1])_tensor([[1],
        [1],
        [0],
        [1],
        [1],
        [1],
        [0],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [0],
        [0],
        [0],
        [0],
        [1],
        [0],
        [0],
        [1],
        [0],
        [0],
        [1],
        [0],
        [1],
        [0],
        [1],
        [0],
        [0],
        [1],
        [0],
        [1],
        [0],
        [0],
        [1],
        [1],
        [0],
        [0],
        [1],
        [0],
        [1],
        [1],
        [1],
        [0],
        [0]])

In [ ]:
# Evaluation

S, info = env.reset()
total_reward = 0
done = False

h.eval()
f.eval()
g.eval()

while not done:
    with torch.no_grad():
        hidden_state = h(torch.tensor(S, dtype=torch.float32))
        policy_logits_pred, value_logits_pred = f(hidden_state)
        
        if use_MCTS_in_evaluation:
            tree = MCTSTree(
                MCTSNode(
                    hidden_state=hidden_state,
                    n_actions=n_actions,
                    p=F.softmax(policy_logits_pred, dim=-1),
                    v = value_logits_pred,
                    identifier="root",
                    ),
                n_simulations=n_MCTS_sims_evaluation,
                dynamics_function=g,
                prediction_function=f,
                n_actions=n_actions
            )
            nu, pi = tree.search()
            A = tree.sample()
        else:
            A = torch.multinomial(F.softmax(policy_logits_pred, dim=-1), 1) # TODO probability tensor contains either `inf`, `nan` or element < 0
    
    S, R, terminated, truncated, info = env.step(A.item())
    total_reward += R
    done = terminated or truncated

print(total_reward)
writer.add_scalar('Eval/Reward', total_reward, i_epoch)

h.train()
f.train()
g.train()


In [ ]:
#
writer.close()